# TP9 — Rendre le modèle réellement utilisable

**Cas d'usage 01 — Certification IA : Résiliation client SaaS (churn)**

| | |
|---|---|
| **Objectif** | Passer du notebook à un outil réellement utilisable : modèles sérialisés, fonction de scoring qui traduit une prédiction en action, architecture cible documentée. |
| **Livrable** | `scripts/train_and_serialize.py`, `scripts/scoring.py`, deux modèles `.joblib`, une architecture cible. |
| **Enjeu** | Un notebook Jupyter, personne ne l'ouvre en entreprise tous les matins — comment connecter ces modèles aux outils que les équipes utilisent déjà ? |
| **Compétences** | C6 (implémenter la solution), C7 (architecture cible) |

> Suite de `TP8.ipynb` (deux modèles finaux validés). **Leçon du projet DL/ML
> appliquée d'emblée** : la logique réutilisable vit dans des scripts `.py`, ce
> notebook ne fait qu'appeler et vérifier — il ne réimplémente rien.

## §1 — Entraînement et sérialisation

`scripts/train_and_serialize.py` reprend exactement les configurations déjà validées
(régression logistique TP5/TP6, Random Forest TP8) et écrit trois artefacts dans
`models/` : les deux modèles sérialisés (`.joblib`) et un `metadata.json` (features,
métriques, seuils).

In [1]:
import subprocess, sys, json
from pathlib import Path

result = subprocess.run(
    [sys.executable, "scripts/train_and_serialize.py"],
    capture_output=True, text=True, cwd=".",
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
print(f"Code retour : {result.returncode}")


[1/3] Chargement des données...
  5,000 lignes
[2/3] Entraînement + sérialisation du modèle churn (régression logistique)...
  ROC-AUC=0.8803  PR-AUC=0.7615
[3/3] Entraînement + sérialisation du modèle CLV (Random Forest)...
  MAE=33441.55€  R²=0.7293

Artefacts écrits dans C:\Users\Aelion\py-init\cas_usage\models/ : model_churn.joblib, model_clv.joblib, metadata.json

Code retour : 0


In [2]:
metadata = json.loads(Path("models/metadata.json").read_text(encoding="utf-8"))
print(json.dumps(metadata, indent=2, ensure_ascii=False))

for f in ["model_churn.joblib", "model_clv.joblib", "metadata.json"]:
    p = Path("models") / f
    print(f"{f:20s} -> {p.stat().st_size:,} octets")


{
  "feature_cols": [
    "secteur",
    "pays",
    "taille_entreprise",
    "plan",
    "anciennete_mois",
    "sieges_souscrits",
    "utilisateurs_actifs",
    "taux_adoption_pct",
    "connexions_30j",
    "heures_usage_30j",
    "fonctionnalites_total",
    "fonctionnalites_utilisees",
    "nb_integrations",
    "derniere_connexion_jours",
    "tickets_support_90j",
    "delai_reponse_support_h",
    "csat",
    "retards_paiement_12m",
    "revenu_mensuel_recurrent_eur",
    "prix_mensuel_par_siege_eur",
    "fonctionnalites_incluses",
    "sla_reponse_h",
    "quota_stockage_go",
    "support_dedie"
  ],
  "categorical_features": [
    "secteur",
    "pays",
    "taille_entreprise",
    "plan",
    "support_dedie"
  ],
  "numeric_features": [
    "anciennete_mois",
    "sieges_souscrits",
    "utilisateurs_actifs",
    "taux_adoption_pct",
    "connexions_30j",
    "heures_usage_30j",
    "fonctionnalites_total",
    "fonctionnalites_utilisees",
    "nb_integrations",
    "derni

## §2 — Vérification : les modèles sérialisés reproduisent-ils TP6/TP8 ?

Un artefact qu'on ne recharge jamais n'est pas vérifié. On recharge les modèles
**depuis le disque** (pas depuis la mémoire du script précédent) et on compare aux
métriques déjà documentées.

In [3]:
import joblib
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

model_churn = joblib.load("models/model_churn.joblib")
model_clv = joblib.load("models/model_clv.joblib")

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username="indusense_user", password="ThEP@ssW0rd",
    host="localhost", port=5432, database="churn_saas_db",
)
engine = create_engine(db_url)
df = pd.read_sql("SELECT * FROM clients_churn", engine)

FEATURE_COLS = metadata["feature_cols"]
X = df[FEATURE_COLS]
y_churn = df["churn"]

_, X_test, _, y_test = train_test_split(X, y_churn, test_size=0.2, stratify=y_churn, random_state=42)
y_prob = model_churn.predict_proba(X_test)[:, 1]

roc_recharge = round(roc_auc_score(y_test, y_prob), 4)
pr_recharge = round(average_precision_score(y_test, y_prob), 4)

print(f"ROC-AUC rechargé : {roc_recharge}  (attendu TP6 : 0.8803)")
print(f"PR-AUC rechargé  : {pr_recharge}  (attendu TP6 : 0.7615)")
print(f"Identique : {roc_recharge == metadata['churn']['roc_auc'] and pr_recharge == metadata['churn']['pr_auc']}")


ROC-AUC rechargé : 0.8803  (attendu TP6 : 0.8803)
PR-AUC rechargé  : 0.7615  (attendu TP6 : 0.7615)
Identique : True


## §3 — Esquisse d'API de scoring

`scripts/scoring.py` traduit une prédiction brute en information actionnable : score de
risque, CLV prédite, **priorité** (matrice risque × CLV, cf. TP1) et **action
recommandée** (règle simple basée sur le signal le plus préoccupant, cf. TP7).

In [4]:
import sys
sys.path.insert(0, "scripts")
from scoring import score_client, load_models

model_churn_s, model_clv_s, metadata_s = load_models()

echantillon = df.sample(6, random_state=7)
resultats = []
for _, row in echantillon.iterrows():
    r = score_client(row, model_churn_s, model_clv_s, metadata_s)
    r["client_id"] = row["client_id"]
    r["churn_reel"] = row["churn"]
    resultats.append(r)

df_resultats = pd.DataFrame(resultats)[
    ["client_id", "proba_churn", "tier_risque", "clv_predite_eur", "tier_clv", "priorite", "action_recommandee", "churn_reel"]
]
df_resultats


,client_id,proba_churn,tier_risque,clv_predite_eur,tier_clv,priorite,action_recommandee,churn_reel
0,CLI-002789,0.283,Modéré,94730.0,Élevée,Surveillance,Proposer un nouvel échéancier (retards de paie...,1
1,CLI-001158,0.374,Modéré,11779.0,Moyenne,Modérée,Proposer un nouvel échéancier (retards de paie...,0
2,CLI-003691,0.916,Élevé,5655.0,Moyenne,Élevée,Proposer un nouvel échéancier (retards de paie...,1
3,CLI-004605,0.992,Élevé,3159.0,Moyenne,Élevée,Proposer une formation / démonstration personn...,1
4,CLI-004445,0.087,Faible,8553.0,Moyenne,Faible,Suivi standard,0
5,CLI-001076,0.040,Faible,13710.0,Moyenne,Faible,Suivi standard,0


## §4 — Exemple d'usage détaillé (cf. exemple du TP1)

In [5]:
exemple = echantillon.iloc[0]
resultat_exemple = score_client(exemple, model_churn_s, model_clv_s, metadata_s)

print(f"Client {exemple['client_id']}")
print(f"  Probabilité de churn : {resultat_exemple['proba_churn']:.1%}  (tier : {resultat_exemple['tier_risque']})")
print(f"  CLV estimée          : {resultat_exemple['clv_predite_eur']:,.0f} €  (tier : {resultat_exemple['tier_clv']})")
print(f"  Priorité             : {resultat_exemple['priorite']}")
print(f"  Action recommandée   : {resultat_exemple['action_recommandee']}")
print(f"  (churn réellement observé : {exemple['churn']})")


Client CLI-002789
  Probabilité de churn : 28.3%  (tier : Modéré)
  CLV estimée          : 94,730 €  (tier : Élevée)
  Priorité             : Surveillance
  Action recommandée   : Proposer un nouvel échéancier (retards de paiement constatés)
  (churn réellement observé : 1)


## §5 — Architecture cible et contraintes

**Batch mensuel, pas de temps réel.** Le churn se joue à l'échéance contractuelle, pas
à la minute — un scoring mensuel de toute la base, aligné sur le cycle de facturation,
suffit largement et évite la complexité d'une inférence temps réel non justifiée par
le besoin métier.

```
┌─────────────────┐     ┌──────────────────────┐     ┌─────────────────────┐
│  Entrepôt/BDD    │     │  Job batch mensuel    │     │  Table de sortie     │
│  (usage,support, │ ──► │  scripts/scoring.py   │ ──► │  scores_churn_clv    │
│  facturation)    │     │  sur toute la base    │     │  (Postgres)          │
└─────────────────┘     └──────────────────────┘     └──────────┬───────────┘
                                                                   │
                                                                   ▼
                                                        ┌─────────────────────┐
                                                        │  Export / connecteur │
                                                        │  vers le CRM (CSM)   │
                                                        └─────────────────────┘
```

**Contraintes identifiées :**

| Contrainte | Réponse |
|---|---|
| Fraîcheur des données | Suffisante à J-30 (cycle mensuel) — pas de flux temps réel à maintenir |
| Latence | Non critique — un batch de quelques minutes sur 5000 comptes est largement acceptable |
| Intégration CRM | Export table/CSV consommé par le CRM existant, pas de nouvelle interface utilisateur à construire |
| Dérive des données | À surveiller dans le temps (TP10) — le modèle est entraîné sur un instantané, l'usage réel évolue |
| Reproductibilité | `scripts/train_and_serialize.py` rejoue tout depuis la base — pas de dépendance à un notebook exécuté à la main |

## Journal de bord — Synthèse TP9

**Fait** :
- Deux modèles finaux entraînés et sérialisés via un script (`train_and_serialize.py`)
  — pas un notebook qui prétendrait être l'artefact de production.
- Vérification par rechargement depuis le disque : métriques identiques à TP6/TP8, pas
  de divergence entre l'objet entraîné et l'objet sérialisé.
- Fonction de scoring (`scoring.py`) qui va au-delà d'une probabilité brute : priorité
  (risque × CLV, TP1) et action recommandée (règle simple sur le signal dominant, TP7).
- Architecture cible documentée : batch mensuel, table Postgres de sortie, export vers
  CRM — pas de temps réel, la fraîcheur mensuelle suffit au besoin métier.

**Reste à faire** :
- TP10 — mesurer l'impact métier en euros (comptes sauvés) et prévoir le plan de
  monitoring / ré-entraînement (amélioration continue).